In [66]:
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from sklearn.metrics import classification_report

from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [4]:
data = load_dataset('cornell-movie-review-data/rotten_tomatoes')

In [5]:
print(f"Training split : {len(data['train'])}")
print(f"Testing split : {len(data['test'])}")
print(f"validation split : {len(data['validation'])}")

Training split : 8530
Testing split : 1066
validation split : 1066


In [47]:
for i in map(int, np.random.randint(0, len(data['train']), 5)):
    print(f"Text {i}: {data['train']['text'][i]}")
    print(f"Label {i}: {data['train']['label'][i]}\n")

Text 5596: made me feel uneasy , even queasy , because [solondz's] cool compassion is on the border of bemused contempt .
Label 5596: 0

Text 2373: some may choose to interpret the film's end as hopeful or optimistic but i think payne is after something darker .
Label 2373: 1

Text 7243: friday after next has the same problem that next friday did -- it's called where's chris tucker when you need him ?
Label 7243: 0

Text 8525: any enjoyment will be hinge from a personal threshold of watching sad but endearing characters do extremely unconventional things .
Label 8525: 0

Text 2356: judith and zaza's extended bedroom sequence . . . is so intimate and sensual and funny and psychologically self-revealing that it makes most of what passes for sex in the movies look like cheap hysterics .
Label 2356: 1



In [8]:
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(model_path,
                                                           device_map ='auto')


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
message_id = 7243
message = [{"role" : "user",
            "content" : data['train'][message_id]['text']}]

prompt = message[0]['content']

inputs = tokenizer(prompt, return_tensors="pt")

for k,v in inputs.items():
  inputs[k] = v.to(model.device)

output = model(**inputs)

probability = torch.softmax(output.logits, dim=-1)
prediction = torch.argmax(probability, dim=-1)

label = 0 if prediction == 0 else 1

print(f"Probabilities : {probability[0].tolist()}")
print(f"Label : {label}")

Probabilities : [0.8385090231895447, 0.1533861756324768, 0.00810484029352665]
Label : 0


In [65]:
y_pred = []

for i, content in enumerate(tqdm(data['test']['text'], total=len(data['test']))):
    try:
        inputs = tokenizer(content, return_tensors="pt")

        for k, v in inputs.items():
            inputs[k] = v.to(model.device)

        with torch.no_grad():
            output = model(**inputs)

        probability = torch.argmax(output.logits, dim=-1).item()

        assignment = 0 if probability == 0 else 1
        y_pred.append(assignment)

    except Exception as e:
        print(f"Error at index {i}: {e}")
        break

100%|██████████| 1066/1066 [00:14<00:00, 73.23it/s]


In [59]:
for i in map(int, np.random.randint(0, len(data['test']), 5)):
    print(f"Text {i}: {data['train']['text'][i]}")
    print(f"Label {i}: {data['train']['label'][i]}\n")

Text 488: few films this year have been as resolute in their emotional nakedness .
Label 488: 1

Text 94: while mcfarlane's animation lifts the film firmly above the level of other coming-of-age films . . . it's also so jarring that it's hard to get back into the boys' story .
Label 94: 1

Text 157: the year's happiest surprise , a movie that deals with a real subject in an always surprising way .
Label 157: 1

Text 900: finds a way to tell a simple story , perhaps the simplest story of all , in a way that seems compelling and even original .
Label 900: 1

Text 175: engagingly captures the maddening and magnetic ebb and flow of friendship .
Label 175: 1



In [67]:
def evaluate(y_true, y_pred):
  performance = classification_report(y_true, y_pred, target_names=['Negative Review', 'Positive Review'])
  print(performance)

In [68]:
evaluate(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.81      0.69      0.75       533
Positive Review       0.73      0.84      0.78       533

       accuracy                           0.77      1066
      macro avg       0.77      0.77      0.77      1066
   weighted avg       0.77      0.77      0.77      1066

